# Funnel 분석용 master 테이블 만들기

**목적**: 주문 완료 → 결제 승인 → 물류 인도 → 배송 완료 → 리뷰 작성 구간별 소요시간을 분석하기 위한 슬림 테이블 (`funnel_master_df.csv`) 생성.

**grain**: 주문 1건 = 1행 (order grain)

**포함 컬럼**
- `orders`: order_id, customer_id, order_status, order_purchase_timestamp, order_approved_at, order_delivered_carrier_date, order_delivered_customer_date, order_estimated_delivery_date
- `customers`: customer_unique_id, customer_state (재구매 추적 + 지역 분석용)
- `order_reviews`: review_score (만족도)

In [1]:
import pandas as pd

# 모두 상대경로 (이 노트북과 같은 폴더에 raw csv 위치)
ORDER_DATE_COLS = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

orders    = pd.read_csv("olist_orders_dataset.csv", parse_dates=ORDER_DATE_COLS)
customers = pd.read_csv("olist_customers_dataset.csv")
reviews   = pd.read_csv("olist_order_reviews_dataset.csv",
                        parse_dates=["review_creation_date"])

print("orders   ", orders.shape)
print("customers", customers.shape)
print("reviews  ", reviews.shape)


orders    (99441, 8)
customers (99441, 5)
reviews   (99224, 7)


In [2]:
# --- reviews 를 order_id 단위로 집계 ----------------------------------------
# 한 주문에 리뷰가 최대 3건까지 있으므로, 가장 빠른 리뷰의 score 1개로 통합
reviews_one = (
    reviews.sort_values("review_creation_date")
           .groupby("order_id", as_index=False)
           .agg(review_score=("review_score", "first"))
)
print("reviews_one:", reviews_one.shape)


reviews_one: (98673, 2)


In [3]:
# --- funnel_master_df 조립 ---------------------------------------------------
# 1) orders 에 customers 의 필요한 컬럼만 left join
# 2) 거기에 reviews_one (1주문=1행) 을 left join
funnel_master_df = (
    orders.merge(
        customers[["customer_id", "customer_unique_id", "customer_state"]],
        on="customer_id", how="left",
    )
    .merge(reviews_one, on="order_id", how="left")
)

# 요청된 컬럼만 명시적으로 선택 (순서도 고정)
FUNNEL_COLS = [
    # orders
    "order_id", "customer_id", "order_status",
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date",
    # customers
    "customer_unique_id", "customer_state",
    # reviews
    "review_score",
]
funnel_master_df = funnel_master_df[FUNNEL_COLS]

# grain 검증: orders 와 행 수 동일해야 함 (1주문=1행)
assert len(funnel_master_df) == len(orders), "order grain 깨짐"
print("funnel_master_df:", funnel_master_df.shape)
funnel_master_df.head()


funnel_master_df: (99441, 11)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_state,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,SP,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,BA,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,GO,5.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,RN,5.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,SP,5.0


In [4]:
# --- 진단 & 저장 -------------------------------------------------------------
print("shape       :", funnel_master_df.shape)
print("order_status:", funnel_master_df["order_status"].value_counts().to_dict())
print()
print("결측 비율:")
print((funnel_master_df.isna().mean() * 100).round(2).astype(str) + " %")

funnel_master_df.to_csv("funnel_master_df.csv", index=False)
print("\n저장 완료 -> funnel_master_df.csv")


shape       : (99441, 11)
order_status: {'delivered': 96478, 'shipped': 1107, 'canceled': 625, 'unavailable': 609, 'invoiced': 314, 'processing': 301, 'created': 5, 'approved': 2}

결측 비율:
order_id                          0.0 %
customer_id                       0.0 %
order_status                      0.0 %
order_purchase_timestamp          0.0 %
order_approved_at                0.16 %
order_delivered_carrier_date     1.79 %
order_delivered_customer_date    2.98 %
order_estimated_delivery_date     0.0 %
customer_unique_id                0.0 %
customer_state                    0.0 %
review_score                     0.77 %
dtype: str

저장 완료 -> funnel_master_df.csv


---

# 📒 코드 설명

## 무엇을 만들었나
**`funnel_master_df.csv`** — 퍼널 분석 전용 슬림 테이블.
- 1행 = 1 주문 (order grain)
- 총 11개 컬럼만 포함 (분석에 꼭 필요한 것만)

## 사용한 원본 3개와 컬럼

| 원본 테이블 | 가져온 컬럼 | 용도 |
|---|---|---|
| `olist_orders_dataset` | `order_id`, `customer_id`, `order_status`, 5개 timestamp | 퍼널 단계별 시각 + 주문 상태 |
| `olist_customers_dataset` | `customer_unique_id`, `customer_state` | 동일 고객 추적, 지역 분석 |
| `olist_order_reviews_dataset` | `review_score` | 만족도 |

## 단계별 처리

### ① 데이터 로드
- `parse_dates=` 로 5개 주문 timestamp를 **datetime 타입으로 바로 변환**.
  → 나중에 분석 노트북에서 `df["a"] - df["b"]` 만으로 구간 시간 계산이 됨.
- reviews 는 집계할 때 가장 빠른 리뷰를 고르기 위해 `review_creation_date` 만 같이 로드 (최종 테이블에는 안 들어감).

### ② reviews 집계
- 한 주문에 리뷰가 **최대 3건** 있어 그대로 merge 하면 행이 늘어남.
- `groupby("order_id") + first` 로 가장 빠른 리뷰의 점수 1개만 남김 → 1주문=1행 보장.

### ③ 조립
```
orders  +  customers (3컬럼만)  +  reviews_one (score 1개)
```
- 모두 **left join** → 원본 99,441개 주문이 빠짐없이 유지됨.
- 마지막에 `FUNNEL_COLS` 리스트로 **요청한 컬럼만 골라 순서 고정**.
- `assert len(funnel_master_df) == len(orders)` 로 행 폭증 없는지 자동 확인.

### ④ 저장
- 같은 폴더에 `funnel_master_df.csv` 로 저장 (상대경로).

## 퍼널 분석할 때 이 테이블로 할 일

1. **status 필터**
   ```python
   df = pd.read_csv("funnel_master_df.csv", parse_dates=[...])
   df = df[df["order_status"] == "delivered"]   # 5단계가 모두 채워진 주문만
   ```
2. **구간별 소요시간 계산** (단위 days)
   | 구간 | 계산식 |
   |---|---|
   | 결제 승인까지 | `order_approved_at − order_purchase_timestamp` |
   | 판매자 → 택배사 | `order_delivered_carrier_date − order_approved_at` |
   | 택배사 → 고객 | `order_delivered_customer_date − order_delivered_carrier_date` |
3. **시각화**: 히스토그램 + 박스플롯 으로 분포 비교 → 어느 구간이 가장 김?
4. **핵심 질문 답하기**: `판매자→택배사` vs `택배사→고객` 의 중앙값/평균 비교
   → "판매자 늦음 vs 택배사 늦음" 판정.

## 주의
- timestamp 컬럼은 csv 로 저장하면 **문자열** 이 되므로, 분석 노트북에서 다시 읽을 때는 반드시 `parse_dates=ORDER_DATE_COLS` 로 읽어야 시간 연산이 됨.
- `review_score` 는 리뷰가 없는 주문은 NaN. 만족도 분석할 땐 dropna 필요.